# Breast cancer prediction from polygenic risk scores

**How to use this notebook**
1. Put `train.csv` and `test.csv` in `data/` (git ignores that folder, so the data is never pushed).
2. Change knobs in the **KNOBS** cell, then *Run All*.
3. Every run is appended to `runs_log.csv` with its knobs and scores, so you can see what helped.
4. Use the **sweep** cell to try many knob values automatically.

**Trust the cross-validation (CV) AUC more than the public leaderboard.** The public test set is small, so leaderboard scores are noisy. Everything that learns from data (feature selection, scaling, PCA, ancestry statistics) is fit inside each training fold, so the CV AUC is an honest estimate.

In [ ]:
%load_ext autoreload
%autoreload 2
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import prs_lib as P
pd.set_option("display.width", 200)

## KNOBS
Everything you might want to tune lives here. Each knob has a comment saying what it does and *why* you'd change it.

In [ ]:
CFG = {
    "run_name": "baseline",        # names the submission file and the log row
    "seed": 42,                    # change to check results aren't luck

    "data": {
        "data_dir": "data",
        "train_file": "train.csv",
        "test_file": "test.csv",
        "target": "breast_cancer",
        "id_col": None,            # None = auto-detect (else row number is used)
        "ancestry_col": None,      # None = auto-detect (name contains "ancestry")
        "fillna": 0.0,             # missing Z-score -> 0 = population average
        "use_cache": True,         # save a pickle in data/ so reloading is fast
        # Speed knobs for quick experiments; set to None for real runs.
        "debug_max_features": None,  # e.g. 500: random subset of PRS columns
        "debug_max_rows": None,      # e.g. 5000: random subset of training rows
    },

    "cv": {
        "n_folds": 5,              # more folds = less noisy estimate, slower
        "max_folds": None,         # e.g. 2: only run the first 2 folds (fast check)
    },

    "preprocess": {
        # Ancestry: PRS are built mostly from European GWAS, so the same Z-score
        # can mean different things in different ancestries. This rescales each
        # PRS to mean 0 / sd 1 *within* each ancestry group.
        "restandardize_by_ancestry": True,
        "min_group_size": 50,           # smaller groups use the global mean/sd
        "standardize_unseen_globally": True,
        "clip_z": 5.0,                  # cap extreme Z-scores (None = off)

        # Feature selection, ranked by |AUC - 0.5| of each PRS alone.
        "top_k_features": 1000,         # keep the strongest K PRS (None = all)
        "min_univariate_strength": None,  # e.g. 0.01: drop PRS with |AUC-0.5| below this
        "corr_threshold": 0.95,         # drop near-duplicate PRS (None = off)

        # Dimensionality reduction
        "pca_components": None,         # e.g. 50 (None = off)
        "pca_mode": "append",           # "append" PCs to PRS, or "replace" PRS with PCs

        # Ancestry features
        "ancestry_onehot": True,        # add 0/1 column per ancestry group
        "interaction_top_k": 0,         # add PRS x ancestry terms for the top K PRS
    },

    "models": {
        # Baseline: the single strongest PRS (picked on each training fold).
        # If a fancier model can't beat this, it's overfitting.
        "best_single": {"enabled": True},

        # Penalized logistic regression: the standard way to combine many PRS.
        "logreg": {
            "enabled": True,
            "C": 0.05,           # smaller = stronger penalty = simpler model
            "l1_ratio": 0.5,     # 0 = ridge (shrink all), 1 = lasso (zero many out)
            "solver": "saga",    # saga supports any l1_ratio
            "max_iter": 2000,
            "tol": 1e-3,
            "scale": True,       # standardize columns before fitting
            "class_weight": None,  # "balanced" reweights cases; AUC rarely cares
        },
        # Gradient-boosted trees: can find non-linear effects and interactions.
        "lgbm": {
            "enabled": True,
            "num_boost_round": 3000,
            "early_stopping_rounds": 150,
            "early_stopping_frac": 0.15,  # slice of training fold used to stop early
            "params": {
                "learning_rate": 0.02,
                "num_leaves": 15,          # tree complexity
                "max_depth": -1,
                "min_data_in_leaf": 50,    # bigger = smoother, less overfit
                "feature_fraction": 0.2,   # share of columns each tree sees
                "bagging_fraction": 0.8,
                "bagging_freq": 1,
                "lambda_l1": 0.0,
                "lambda_l2": 1.0,
                "num_threads": -1,
            },
        },
    },

    "ensemble": {
        "method": "rank_mean",           # "rank_mean" (good for AUC) or "mean"
        "weights": {"best_single": 0.0, "logreg": 1.0, "lgbm": 1.0},  # 0 = leave out
    },

    "output": {
        "submission_dir": "submissions",
        "id_col_name": "id",             # match Kaggle's sample_submission header
        "pred_col_name": "breast_cancer",
        "which": "ensemble",             # or "logreg" / "lgbm"
    },
}

## Load data

In [ ]:
data = P.load_data(CFG)

## Explore
How common are cases, what ancestry groups exist, and which PRS predict well on their own? The per-ancestry AUC columns show whether a PRS works equally well in every group.

In [ ]:
print(pd.crosstab(data["a"], data["y"], margins=True).rename(columns={0: "controls", 1: "cases"}))
uni = P.univariate_table(data)
uni.head(20)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(uni["auc"], bins=80)
ax[0].axvline(0.5, color="k", lw=0.8)
ax[0].set(title="AUC of each PRS on its own", xlabel="AUC", ylabel="# PRS")
ax[1].plot(uni["strength"].to_numpy())
ax[1].set(title="Ranked strength |AUC - 0.5|", xlabel="rank", ylabel="strength", xscale="log")
plt.tight_layout()

## Train with cross-validation

In [ ]:
res = P.run_cv(CFG, data)
P.log_run(res, note="")   # add a note to remember what you changed
res["by_ancestry"].round(4)

In [ ]:
pd.DataFrame(res["fold_auc"]).round(4)   # one row per fold

## Write the submission

In [ ]:
P.write_submission(res, data, CFG, which=CFG["output"]["which"])

## Sweep: try many knob values
Keys use dots to reach nested knobs. Every combination is run and logged.
Tip: turn off the model you're *not* tuning, and use `cv.max_folds` / `debug_*` knobs for speed, then confirm the winner with a full run.

In [ ]:
RUN_SWEEP = False   # flip to True to run

if RUN_SWEEP:
    base = copy.deepcopy(CFG)
    base["run_name"] = "sweep_logreg"
    base["models"]["lgbm"]["enabled"] = False
    grid = {
        "models.logreg.C": [0.01, 0.03, 0.1],
        "models.logreg.l1_ratio": [0.2, 0.5, 0.9],
        "preprocess.top_k_features": [500, 2000],
    }
    sweep_res = P.sweep(base, data, grid)
    display(sweep_res)

## Compare all runs so far

In [ ]:
log = pd.read_csv("runs_log.csv")
show = ["time", "run_name", "note"] + [c for c in log.columns if c.startswith("auc_")]
log.sort_values("auc_ensemble", ascending=False)[show].head(20)